# 01 Function Calling

让 LLM 调用外部工具——它不生成文本，而是决定"该调用哪个函数"。

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.client import get_client

client = get_client()

## 定义工具

In [ ]:
def get_weather(city):
    weather_data = {
        "北京": {"temp": 25, "desc": "晴朗"},
        "上海": {"temp": 28, "desc": "多云"},
        "广州": {"temp": 32, "desc": "雷阵雨"},
    }
    w = weather_data.get(city, {"temp": 20, "desc": "未知"})
    return json.dumps(w, ensure_ascii=False)

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的天气",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "城市名，如 北京"},
            },
            "required": ["city"],
        },
    },
}

## 调用：LLM 决定调工具

In [ ]:
messages = [{"role": "user", "content": "北京今天天气怎么样？"}]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=[weather_tool],
)

msg = response.choices[0].message
print(f"有 tool_calls: {msg.tool_calls is not None}")
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"工具名: {tc.function.name}")
    print(f"参数: {tc.function.arguments}")

## 执行工具 → 回传结果 → 生成回答

In [ ]:
tc = response.choices[0].message.tool_calls[0]

args = json.loads(tc.function.arguments)
result = get_weather(**args)

messages.append(msg)
messages.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": result,
})

final = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

print(final.choices[0].message.content)

## 查看完整的 messages 链路

In [ ]:
for i, m in enumerate(messages):
    if isinstance(m, dict):
        role = m["role"]
        content = str(m.get("content", ""))[:80]
    else:
        role = m.role
        content = str(getattr(m, "content", ""))[:80]
        if hasattr(m, "tool_calls") and m.tool_calls:
            content += f" [tool_call: {m.tool_calls[0].function.name}]"
    print(f"[{i}] {role}: {content}")